# Lab 0: Workshop Setup

This notebook provisions the **shared endpoint** used across all labs in this workshop.

**Architecture:**
- One `ml.g5.12xlarge` endpoint is created here and kept running for the duration of the workshop
- Each lab deploys its own **Inference Component** (IC) onto this shared endpoint
- Each lab cleans up its IC when done — the endpoint stays
- Run the **Cleanup** cell at the bottom of this notebook only after completing all labs

**Why this pattern?**
- Endpoint provisioning takes 3-5 minutes — doing it once saves ~15 min across labs
- ICs can be swapped in seconds (vs. reprovisioning an entire endpoint)
- This mirrors production best practices: decouple compute from models

> ⚠️ **Cost**: The `ml.g5.12xlarge` instance costs ~$7.09/hour. Remember to run cleanup when done.


In [ ]:
%pip install --upgrade boto3 botocore --quiet


In [ ]:
import json
import time
import os
import boto3
from datetime import datetime

import sys
sys.path.append("..")
from utils import wait_for_endpoint, get_execution_role

role = get_execution_role()
region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account_id}"
timestamp = datetime.now().strftime("%m%d%H%M%S")

sm_client = boto3.client("sagemaker", region_name=region)

print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Bucket:  {bucket}")


## Create the Shared Endpoint

We create an **IC-enabled endpoint** — no model is loaded yet. Each lab will attach its own
Inference Component to deploy a model onto this compute.


In [ ]:
# ---------------------------------------------------------------
# Create the shared workshop endpoint
# ---------------------------------------------------------------
instance_type = "ml.g5.12xlarge"
endpoint_config_name = f"workshop-bench-{timestamp}"
endpoint_name = endpoint_config_name

# Create endpoint config (IC-enabled — no model attached)
sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "InstanceType": instance_type,
        "InitialInstanceCount": 1,
        "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
    }],
)
print(f"✅ Endpoint config '{endpoint_config_name}' created")

# Create endpoint (provisions compute)
sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"⏳ Endpoint '{endpoint_name}' provisioning (~3-5 min)...")

wait_for_endpoint(endpoint_name, region=region)


In [ ]:
# Save endpoint info for other labs to load
endpoint_info = {
    "endpoint_name": endpoint_name,
    "endpoint_config_name": endpoint_config_name,
    "instance_type": instance_type,
    "region": region,
    "role": role,
    "bucket": bucket,
}

os.makedirs("../results", exist_ok=True)
with open("../results/endpoint_info.json", "w") as f:
    json.dump(endpoint_info, f, indent=2)

print(f"\n{'='*60}")
print(f"  WORKSHOP ENDPOINT READY")
print(f"{'='*60}")
print(f"  Endpoint name:  {endpoint_name}")
print(f"  Instance type:  {instance_type}")
print(f"  Region:         {region}")
print(f"{'='*60}")
print(f"\n✅ Saved to ../results/endpoint_info.json")
print(f"   Labs 1-4 will load this automatically.")


---

## ✅ Setup Complete — Proceed to Lab 1

The endpoint is running. Open `../lab1/lab1_deploy_and_benchmark.ipynb` to start.

---

## 🧹 Final Cleanup (run AFTER all labs are done)

> ⚠️ **Only run the cell below after you've finished ALL labs (1-4).**
> The endpoint will be deleted and cannot be recovered.


In [ ]:
# ---------------------------------------------------------------
# FINAL CLEANUP — Run only after completing all labs
# ---------------------------------------------------------------
# Load endpoint info in case kernel was restarted
with open("../results/endpoint_info.json", "r") as f:
    endpoint_info = json.load(f)

endpoint_name = endpoint_info["endpoint_name"]
endpoint_config_name = endpoint_info["endpoint_config_name"]

sm_client.delete_endpoint(EndpointName=endpoint_name)
print(f"⏳ Endpoint '{endpoint_name}' deleting...")

sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
print(f"✅ Endpoint config '{endpoint_config_name}' deleted")

print(f"\n✅ Workshop endpoint cleaned up. You're done!")
